In [1]:
import cv2
from paho.mqtt import client as mqtt_client
import numpy as np
import tools
import random
from yolox.tracker.byte_tracker import BYTETracker
import argparse
import pandas as pd
import torch



/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#MQTT Broker settings

broker = "emqx1.emqx.io"
port = 1883
topic_1 = "bbox/topic"
topic_2 = "frame/topic"
client_id = f"python-mqtt-{random.randint(0, 100)}"
client = mqtt_client.Client(client_id)

client.connect(broker, port)
client.loop_start()

# Subscribe to topics
bbox_and_labels = tools.on_connect(client, topic=topic_1)
frame = tools.on_connect(client, topic=topic_2)
# Bounding Boxes and labels
bbox_and_labels = tools.on_message(client, bbox_and_labels)
# Frame
frame = tools.on_message(client, frame)

# Zu Testzwecken ist standardmäßig die Videoanalyse innerhalb des Containers eingestellt und bezieht Bboxes und Labels aus der CSV-Eingabe
parser = argparse.ArgumentParser(description="ByteTracker Tracking")
parser.add_argument(
    "--purpose", type=str, default="testing", help="Are you testing or running?"
)
parser.add_argument(
    "-f", "--video", type=str, default="input/Test.mp4", help="Path to video file"
)
parser.add_argument(
    "-c", "--csv", type=str, default="input/output.csv", help="Path to CSV file with bboxes"
)
parser.add_argument("--output_path", type=str, default="output/tracker_output.mp4", help="Path to output video file")

args = parser.parse_args()

purpose = args.purpose
output_path = args.output_path
video_path = args.video
csv = args.csv

In [5]:
# Beispielwerte für die Argumente des Trackers
tracker_args = argparse.Namespace(
    track_thresh=0.29,       # Beispielwert für den Tracking-Schwellenwert
    track_buffer=15,        # Beispielwert für den Puffer
    mot20=False,            # Beispielwert für MOT20
    match_thresh=0.7        # Beispielwert für den Matching-Schwellenwert
)


# Argumente direkt definieren
purpose = "testing"
video_path = "input/Test.mp4"
csv = 'input/Interpolierte_CSV.csv' #"input/output_data.csv" 
output_path = "output/tracker_output.mp4"

tracker = BYTETracker(tracker_args)

if purpose == "testing":
    # Read the video file
    cap = cv2.VideoCapture(video_path)
    # Einlesen der CSV-Datei
    data = pd.read_csv(csv, sep=",") #sep ändern, falls nötig

    

    # Videoeigenschaften abrufen
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec für das Ausgabevideo
    img_size = (width, height)
    out = cv2.VideoWriter(output_path, fourcc, fps, img_size)
    # Initialize the tracker
    frame_count = 1 #508
    i = 1

    # Debugging: Überprüfen von img_size
    print(f"img_size: {img_size}, Typ: {type(img_size)}")
    print(f"width: {width}, Typ: {type(width)}")
    print(f"height: {height}, Typ: {type(height)}")

    while cap.isOpened():
        ret, frame = cap.read()
        # print(f"{frame.shape[0]}, {frame.shape[1]}")
        # print(f"Typ von frame: {type(frame)}")
        # first = data.iloc[i, 0]
        # print(f"erste Zeile:{first}")
        if not ret or frame is None:
            print("End of video or unable to read the frame.")
            break
        if i >= len(data):
            print("All data processed.")
            break
        if frame_count == data.iloc[i, 0]:
            print("Frame: ", frame_count)
            confidence = data.iloc[i, 1]
            x_min = data.iloc[i, 2]
            y_min = data.iloc[i, 3]
            x_max = data.iloc[i, 4]
            y_max = data.iloc[i, 5]
            label = str(data.iloc[i, 6])
            print(f"Label: {label}, Confidence: {confidence}")
            bbox = [x_min, y_min, x_max, y_max]
            bbox_and_confidence = torch.tensor([[*bbox, confidence]])
            shape = bbox_and_confidence.shape[1]
            print(f"Shape: {shape}")
            img_info = (frame.shape[1], frame.shape[0])
            scale = min(img_size[0] / float(frame.shape[1]), img_size[1] / float(frame.shape[0]))
            print(f"Scale: {scale}")
            online_targets = tracker.update(bbox_and_confidence, img_info, img_size, label)
            # print(f"Tracked Stracks: {len(online_targets.tracked_stracks)}")
            # print(f"Lost Stracks: {len(online_targets.self.lost_stracks)}")
            # print(f"Detections: {len(online_targets.detections)}")
            print(f"Online targets: {online_targets}")
            # online_targets enthält eine Liste von Objekten mit Bounding Boxes und IDs
            for target in online_targets:
                # Extrahiere die Bounding Box und die ID
                print(target.track_id, target.label)
                print(f"Label:{label}")  # ID des Tracks

                # Konvertieren der Bounding Box in (x1, y1, x2, y2)
                bbox = target.tlwh
                x1, y1, w, h = bbox
                x2, y2 = int(x1 + w), int(y1 + h)
                x1, y1 = int(x1), int(y1)

                # Zeichnen der Bounding Box
                color = (0, 255, 0)  
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                # Zeichnen der Track-ID
                text = f"ID: {target.label} (OT_{target.track_id})"
                cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 2, color, 2)

            
            out.write(frame)
            frame_count += 1
            i += 1
        else:
            out.write(frame)
            frame_count += 1


    cap.release()
    out.release()
    

img_size: (1440, 1080), Typ: <class 'tuple'>
width: 1440, Typ: <class 'int'>
height: 1080, Typ: <class 'int'>
Frame:  506
Label: car, Confidence: 0.3
Shape: 5
Scale: 1.0
dists []
dists nach mot20 []
matches [] u_track () u_detection (0,)
Online targets: [OT_119_(1-1)]
119 car
Label:car
Frame:  507
Label: car, Confidence: 0.3
Shape: 5
Scale: 1.0
dists [[0.14045118]]
dists nach mot20 [[0.74213535]]
matches [] u_track [0] u_detection [0]
Online targets: []
Frame:  508
Label: car, Confidence: 0.45956123
Shape: 5
Scale: 1.0
dists [[0.25520428]]
dists nach mot20 [[0.65772076]]
matches [[0 0]] u_track [] u_detection []
Online targets: [OT_119_(1-3)]
119 car
Label:car
Frame:  509
Label: car, Confidence: 0.3
Shape: 5
Scale: 1.0
dists [[0.11578654]]
dists nach mot20 [[0.73473596]]
matches [] u_track [0] u_detection [0]
Online targets: []
Frame:  510
Label: car, Confidence: 0.3
Shape: 5
Scale: 1.0
dists [[0.21183435]]
dists nach mot20 [[0.76355031]]
matches [] u_track [0] u_detection [0]
Online t

/workspace/ByteTrack/yolox/tracker/byte_tracker.py:183: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at  ../aten/src/ATen/native/IndexingUtils.h:30.)
  dets_second = bboxes[inds_second]
/workspace/ByteTrack/yolox/tracker/byte_tracker.py:187: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at  ../aten/src/ATen/native/IndexingUtils.h:30.)
  scores_second = scores[inds_second]


Online targets: []
Frame:  536
Label: car, Confidence: 0.3
Shape: 5
Scale: 1.0
dists []
dists nach mot20 []
matches [] u_track () u_detection (0,)
Online targets: []
Frame:  537
Label: car, Confidence: 0.45519865
Shape: 5
Scale: 1.0
dists []
dists nach mot20 []
matches [] u_track () u_detection (0,)
Online targets: [OT_144_(31-32)]
144 car
Label:car
Frame:  538
Label: car, Confidence: 0.53728414
Shape: 5
Scale: 1.0
dists [[0.09488916]]
dists nach mot20 [[0.5136983]]
matches [[0 0]] u_track [] u_detection []
Online targets: [OT_144_(31-33)]
144 car
Label:car
Frame:  539
Label: car, Confidence: 0.55084896
Shape: 5
Scale: 1.0
dists [[0.11459705]]
dists nach mot20 [[0.5122767]]
matches [[0 0]] u_track [] u_detection []
Online targets: [OT_144_(31-34)]
144 car
Label:car
Frame:  540
Label: car, Confidence: 0.5448014
Shape: 5
Scale: 1.0
dists [[0.10249163]]
dists nach mot20 [[0.51103618]]
matches [[0 0]] u_track [] u_detection []
Online targets: [OT_144_(31-35)]
144 car
Label:car
Frame:  541
